In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import pandas as pd
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RainfallHistogram"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import FigurePlotting_Class

In [ ]:
#Setup

# Region = "TRACER"; Case = "WET"; spinup_hours = "0"
Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"
# Region = "PRECIP"; Case = "WET"; spinup_hours = "12"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_DataSubsetting import DataSubsetting_Class

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
####################################
#GETTING PRECIP DATA

In [ ]:
def ReadPrecipData(yearmonth, inputDirectory):
    inputPath = os.path.join(inputDirectory,f"ncdd-{yearmonth}-grd-scaled.nc")
    precipData = xr.open_dataset(inputPath)['prcp']
    return precipData
    
def SubsetData_Time(data,yearmonthday):
    data_T = data.sel(time=yearmonthday)
    # print(data_T.time) #testing
    return data_T
    
def GetAccumulatedPrecipData(ModelData): 
    inputDirectory = os.path.join(DirectoryManager.dataDirectory,
                                  f"Observation_Data/{ModelData.region}/nClimGrid_PrecipData")
    print(f"reading from {inputDirectory}")
    yearmonthdays = [d.replace('-','') for d in ModelData.simulationDates][0:-1]

    print(f"for dates {yearmonthdays}")
    for count,yearmonthday in enumerate(yearmonthdays):
        yearmonth = yearmonthday[0:6]
        precipData = ReadPrecipData(yearmonth,inputDirectory)
        if count == 0:
            precipData_T = SubsetData_Time(precipData,yearmonthday)
        else:
            precipData_T += SubsetData_Time(precipData,yearmonthday)
    return precipData_T

In [ ]:
precipData_T = GetAccumulatedPrecipData(ModelData_NSSL)   
precipData_T_Subset = DataSubsetting_Class.SubsetDataRegion(precipData_T, ModelData_NSSL)

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName):
    """
    Retrieves a variable subset from the given model data.
    If varName contains a '+', returns the sum of the two variables.
    """
    if '+' in varName:
        var1, var2 = varName.split('+')
        var1 = var1.strip()
        var2 = var2.strip()

        subset1 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var1)
        subset2 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var2)
        variableSubset = subset1 + subset2
    else:
        variableSubset = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                             dataSubset_diag, dataSubset_static, varName)

    return variableSubset

def RunCalculations(ModelData, varNames):
    outputDictionary={}
    
    t = ModelData.Ntime-1
        
    #Loading Data
    [dataSubset, dataSubset_diag, dataSubset_static, lat, lon, _, _] = DataOperator_Class.GetData_Subset(ModelData, t)

    for varName in varNames:
        print(f"Running for {varName}")
        #Subsetting Data

        variableSubset = GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName)
        
    return variableSubset, variableSubset.latitude, variableSubset.longitude

In [ ]:
####################################
#CALCULATION

In [ ]:
rain_NSSL, lat,lon = RunCalculations(ModelData=ModelData_NSSL,varNames=['rainnc+rainc'])
rain_TEMPO, _,_ = RunCalculations(ModelData=ModelData_TEMPO,varNames=['rainnc+rainc'])
clim = min(rain_NSSL.min(),rain_TEMPO.min()),max(rain_NSSL.max(),rain_TEMPO.max())

In [ ]:
#Applying Radar Mask
RadarDataMask = RadarObservationMask_Class.LoadMaskData_MRMS(DirectoryManager, ModelData_NSSL)
RadarDataMask_interp = (RadarDataMask*1).interp(
    latitude = precipData_T_Subset.lat,
    longitude = precipData_T_Subset.lon,
    method="nearest"
)
RadarDataMask_interp = RadarDataMask_interp == 1

rain_NSSL = rain_NSSL.where(RadarDataMask == True).data
rain_TEMPO = rain_TEMPO.where(RadarDataMask == True).data
precipData_T_Subset = precipData_T_Subset.where(RadarDataMask_interp == True)

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
def CreateFigure_2x2(figsize=(10, 10),
                     wspace=0.3, hspace=0.3,
                     left=0.05, right=0.95, top=0.92, bottom=0.08):
    """
    Creates a 2x2 grid of subplots with adjustable layout.
    Returns (fig, axes) where axes is a 2D list [[ax00, ax01], [ax10, ax11]].
    """
    fig = plt.figure(figsize=figsize)
    fig.subplots_adjust(left=left, right=right, top=top, bottom=bottom,
                        wspace=wspace, hspace=hspace)

    gs = gridspec.GridSpec(2, 2, figure=fig)

    ax00 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
    ax01 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())
    ax10 = fig.add_subplot(gs[1, 0])
    ax11 = fig.add_subplot(gs[1, 1])

    axes = [[ax00, ax01],
            [ax10, ax11]]

    return fig, axes

def CreateFigure_2x1Combo(figsize=(10, 10),
                          wspace=0.35, hspace=0.1,
                          left=0.07, right=0.95, top=0.92, bottom=0.08):
    """
    Creates a figure with two subplots on top (map style)
    and one wide subplot spanning the full bottom row.
    
    Layout:
        +-----------+-----------+
        |   ax00    |   ax01    |
        +-----------------------+
        |        ax_bottom      |
        +-----------------------+
    
    Returns (fig, axes) where:
      axes = [[ax00, ax01], [ax_bottom]]
    """
    fig = plt.figure(figsize=figsize)
    fig.subplots_adjust(left=left, right=right, top=top, bottom=bottom,
                        wspace=wspace, hspace=hspace)

    gs = gridspec.GridSpec(2, 2, figure=fig, height_ratios=[1, 0.8])

    # Two map panels on top
    ax00 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
    ax01 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())

    # One wide histogram panel across bottom
    ax_bottom = fig.add_subplot(gs[1, :])  # spans both columns

    axes = [[ax00, ax01], [ax_bottom]]

    return fig, axes

def CreateFigure_3x1Combo(figsize=(14, 10),
                          wspace=0.25, hspace=0.15,
                          left=0.06, right=0.97, top=0.93, bottom=0.08):
    """
    Layout:
        +-----------+-----------+-----------+
        |   ax00    |   ax01    |   ax02    |
        +-----------------------------------+
        |             ax_bottom              |
        +-----------------------------------+

    Returns (fig, axes) where:
      axes = [[ax00, ax01, ax02], [ax_bottom]]
    """

    fig = plt.figure(figsize=figsize)
    fig.subplots_adjust(left=left, right=right, top=top, bottom=bottom,
                        wspace=wspace, hspace=hspace)

    # 2 rows × 3 columns grid
    gs = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1, 0.8])

    # Top row: 3 map panels
    ax00 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
    ax01 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())
    ax02 = fig.add_subplot(gs[0, 2], projection=ccrs.PlateCarree())

    # Bottom row: spans all 3 columns
    ax_bottom = fig.add_subplot(gs[1, :])  # slice all columns

    axes = [[ax00, ax01, ax02], [ax_bottom]]

    return fig, axes

In [ ]:
#CONTOUR PLOTTING FUNCTION
from mpl_toolkits.axes_grid1 import make_axes_locatable
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

# Preload map features once (global)
COAST = cfeature.COASTLINE.with_scale("50m")
BORDERS = cfeature.BORDERS.with_scale("50m")
STATES = cfeature.STATES.with_scale("50m")
LAND = cfeature.LAND.with_scale("50m")
LAKES = cfeature.LAKES.with_scale("50m")

def PlotVariable_with_Borders(axis, 
                              variable, varName, lat, lon, multiplier=1, 
                              clim=(None, None), norm=None, cmap="viridis",
                              title=None, units=None,
                              center_colorbar=False):
    """
    Plot a uxarray/xarray variable on a Cartopy map with coastlines, borders, and states.
    Includes a flush colorbar to the right of the axis (Cartopy-safe).
    """

    # --- Handle color limits ---
    vmin = np.nanmin(variable) if clim[0] is None else clim[0]
    vmax = np.nanmax(variable) if clim[1] is None else clim[1]
    clim = (vmin, vmax)

    num_levels = 19
    levels = multiplier * np.linspace(clim[0], clim[1], num_levels)

    # --- Optional centered colorbar ---
    if center_colorbar:
        cmap = "RdBu_r"
        vmax = clim[1]
        vmin = clim[0]
        norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
        levels = multiplier * np.linspace(vmin, vmax, num_levels)

    matrix = multiplier * variable

    # --- Reflectivity special case ---
    if varName in ["refl10cm", "refl10cm_1km"]:
        extend = 'both'
        cmap, norm, levels, ticks = RadarPlotting_Class.GetReflectivityColormap()
        matrix[matrix <= 0] = np.nan
    else:
        extend = None

    # --- Contour plot ---
    im = axis.contourf(
        lon, lat, matrix,
        levels=levels,
        cmap=cmap,
        norm=norm,
        transform=ccrs.PlateCarree(),
        extend=extend
    )

    # --- Add map features ---
    axis.add_feature(COAST, linewidth=1)
    axis.add_feature(BORDERS, linewidth=0.8)
    axis.add_feature(STATES, linewidth=0.5)
    axis.add_feature(LAND, facecolor="lightgray", alpha=0.3)
    axis.add_feature(LAKES, edgecolor="k", facecolor="none")

    # --- Flush colorbar (Cartopy-safe) ---
    label = f"{varName} ({units})" if units else varName
    divider = make_axes_locatable(axis)
    #force a regular Matplotlib axis for colorbar (avoids projection error)
    cax = divider.append_axes("right", size="3%", pad=0, axes_class=plt.Axes)
    cbar = plt.colorbar(im, cax=cax)
    cbar.set_label(label, fontsize=11)
    cbar.ax.tick_params(labelsize=10)

    # --- Labels, ticks, and extent ---
    axis.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=ccrs.PlateCarree())
    xticks = np.round(np.linspace(lon.min(), lon.max(), 5), 1)
    yticks = np.round(np.linspace(lat.min(), lat.max(), 5), 1)
    axis.set_xticks(xticks, crs=ccrs.PlateCarree())
    axis.set_yticks(yticks, crs=ccrs.PlateCarree())

    axis.set_xlabel("Longitude (°E)")
    axis.set_ylabel("Latitude (°N)")
    if title:
        axis.set_title(title)

    return im


In [ ]:
def SaveFigure(fig, ModelData1, ModelData2, dpi=150):
    """
    Saves a figure to the appropriate directory based on the models in combinedDict.
    """
    # --- Define output subdirectory and file path ---
    outputSubDirectory = f"{ModelData1.region}_{ModelData1.case}_{ModelData1.mpType}vs{ModelData2.mpType}_{ModelData1.spinup_hours}hrs"
    os.makedirs(os.path.join(outputPlottingDirectory, outputSubDirectory), exist_ok=True)

    outputFilePath = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory,
        f"RainfallHistogram.png"
    )

    # # --- Save figure ---
    # FigurePlotting_Class.SaveUniformFigure(fig, outputFilePath)
    fig.savefig(outputFilePath, dpi=dpi, bbox_inches=None)
    plt.close(fig)
    print(f"Saved image: {outputFilePath}")

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
def PlotRainContours(axes, rain_NSSL, rain_TEMPO, lat, lon, clim,
                     ModelData_NSSL, ModelData_TEMPO,
                     precipData_T_Subset):
    """
    Plot spatial rain accumulation maps for NSSL and TEMPO on two axes.
    """
    # NSSL contour
    axis = axes[0][0]
    PlotVariable_with_Borders(axis, rain_NSSL, "", lat, lon, clim=clim)
    axis.set_xlabel('latitude'); axis.set_ylabel('longitude')
    axis.set_title(f'{ModelData_NSSL.region}_{ModelData_NSSL.case}_{ModelData_NSSL.mpType} rainnc+rainc')

    # NOAAnClimGrid contour
    axis = axes[0][1]
    PlotVariable_with_Borders(axis, precipData_T_Subset, "", precipData_T_Subset.lat, precipData_T_Subset.lon, clim=clim)
    axis.set_xlabel('latitude'); axis.set_ylabel('longitude')
    axis.set_title(f'NOAA nClimGrid 3-day Accumulated Precip')
    axis.set_ylabel("")                # remove label

    # TEMPO contour
    axis = axes[0][2]
    PlotVariable_with_Borders(axis, rain_TEMPO, "", lat, lon, clim=clim)
    axis.set_xlabel('latitude'); axis.set_ylabel('longitude')
    axis.set_title(f'{ModelData_TEMPO.region}_{ModelData_TEMPO.case}_{ModelData_TEMPO.mpType} rainnc+rainc')
    axis.set_ylabel("")                # remove label

In [ ]:
def PlotRainHistograms(axes, rain_NSSL, rain_TEMPO, precipData_T_Subset):
    """
    Plot outlined histograms of accumulated precipitation for NSSL and TEMPO
    on a single combined bottom axis using shared bins and different colors.
    """
    # Define the bottom axis (only one now)
    axis = axes[1][0]

    # Getting valid non-nan data
    valid_NSSL  = rain_NSSL.flatten()
    valid_NSSL  = valid_NSSL[~np.isnan(valid_NSSL)]
    
    valid_TEMPO = rain_TEMPO.flatten()
    valid_TEMPO = valid_TEMPO[~np.isnan(valid_TEMPO)]
    
    valid_NOAA  = precipData_T_Subset.data.flatten()
    valid_NOAA  = valid_NOAA[~np.isnan(valid_NOAA)]

    # Common bins for fair comparison
    combinedData = np.concatenate([valid_NSSL, valid_TEMPO, valid_NOAA])
    binEdges = np.linspace(combinedData.min(), combinedData.max(), 51)  # 50 bins

    # NSSL histogram (outlined)
    label = f'{ModelData_NSSL.region}_{ModelData_NSSL.case}_{ModelData_NSSL.mpType}'
    axis.hist(valid_NSSL, bins=binEdges,
              histtype='step', linewidth=1.8, color='steelblue', label=label)

    # NOAAnClimGrid histogram (outlined)
    label = f'NOAA nClimGrid Accumulated Precip'
    axis.hist(valid_NOAA, bins=binEdges,
              histtype='step', linewidth=1.8, color='green', label=label)

    # TEMPO histogram (outlined)
    label = f'{ModelData_TEMPO.region}_{ModelData_TEMPO.case}_{ModelData_TEMPO.mpType}'
    axis.hist(valid_TEMPO, bins=binEdges,
              histtype='step', linewidth=1.8, color='darkorange', label=label)

    # Log scale, labels, and limits
    axis.set_yscale("log")
    axis.set_xlim(left=0, right=np.nanmax(combinedData))
    axis.set_xlabel('rainnc + rainc (accumulated precipitation) (mm)')
    axis.set_ylabel('count')

    # Add legend
    axis.legend(frameon=False, fontsize=12)


In [ ]:
####################################
#PLOTTING

In [ ]:
# fig, axes = CreateFigure_2x1Combo(figsize=(10,10))
fig, axes = CreateFigure_3x1Combo(figsize=(15,10))

PlotRainContours(axes, rain_NSSL, rain_TEMPO, lat, lon, clim,
                 ModelData_NSSL, ModelData_TEMPO,
                 precipData_T_Subset)

PlotRainHistograms(axes, rain_NSSL, rain_TEMPO, precipData_T_Subset)

SaveFigure(fig, ModelData1=ModelData_NSSL, ModelData2=ModelData_TEMPO)